# Offline GEO RAG — Colab Edition (Production Ready)

## For Google Colab with GPU

This notebook uses:
- **7B VLM** (better quality than 3B, Colab has enough VRAM)
- **4-bit quantization** (efficient, ~8 GB memory)
- **Google Drive** for persistent storage
- **Vision + Text modes** (proper multimodal)
- **Cached image descriptions** (no re-processing)

## What works here

✓ Images extracted from PDFs and saved  
✓ VLM describes images when retrieved (not during loading)  
✓ Text-only and vision modes working  
✓ Incremental PDF updates  
✓ Production-ready error handling  

## Steps to run

1. Mount Google Drive (Step 0)
2. Install packages (Step 1)
3. Load PDFs (Step 5-7)
4. Build indexes (Step 8-10)
5. Load VLM (Step 11)
6. Run queries (Step 14+)

All files saved to Google Drive — persistent across sessions.


---
## Step 0 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

import os
os.chdir('/content/gdrive/MyDrive')

print("✓ Google Drive mounted")
print("   Working directory:", os.getcwd())

Mounted at /content/gdrive
✓ Google Drive mounted
   Working directory: /content/gdrive/MyDrive


---
## Step 1 — Install packages

In [2]:
# Quick fix — downgrade requests to match google-colab requirement
# !pip install requests==2.32.5 -q


In [3]:
!pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface \
    faiss-cpu sentence-transformers transformers accelerate torch tqdm \
    pymupdf4llm pymupdf rank_bm25 rouge-score Pillow bitsandbytes

print("✓ All packages installed")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts

In [4]:
!pip install requests==2.32.5 -q

print("✓ All dependencies fixed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
✓ All dependencies fixed


In [7]:
import zipfile

archive_path = '/content/Archive.zip'
extraction_path = settings.data_dir

extraction_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(archive_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"✓ Archive '{archive_path}' extracted to '{extraction_path}'")

✓ Archive '/content/Archive.zip' extracted to '/content/data'


---
## Step 2 — Imports

In [5]:
from __future__ import annotations

import gc
import hashlib
import io
import json
import pickle
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import torch
from PIL import Image
from tqdm.auto import tqdm

import fitz
import pymupdf4llm

from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sentence_transformers import CrossEncoder
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

print("✓ All imports done")
print(f"✓ GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

/tmp/ipykernel_1156/1574413597.py:21: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


✓ All imports done
✓ GPU available: True
  GPU: Tesla T4
  Memory: 15.6 GB


---
## Step 3 — Settings

In [8]:
@dataclass
class Settings:
    # Paths — using local /content/data instead of Google Drive
    data_dir:    Path = Path("/content/data")
    storage_dir: Path = Path("/content/storage")
    images_dir:  Path = Path("/content/storage/images")

    # Regions
    geos: Tuple[str, ...] = ("APAC", "EMEA", "AMER")

    # Chunking
    chunk_size:    int = 800
    chunk_overlap: int = 100

    # Retrieval
    top_k:             int             = 3
    retrieval_fetch_k: int             = 12
    score_threshold:   Optional[float] = 1.2
    bm25_top_k:        int             = 5

    # Embedding
    embedding_model:  str = "BAAI/bge-large-en-v1.5"
    bge_query_prefix: str = "Represent this sentence for searching relevant passages: "

    # Reranker
    reranker_model: str = "BAAI/bge-reranker-v2-m3"

    # VLM — 7B for Colab
    vlm_model: str = "Qwen/Qwen2.5-VL-7B-Instruct"

    # Generation
    max_new_tokens:    int   = 512
    image_max_tokens:  int   = 600
    do_sample:         bool  = False
    temperature:       float = 0.7

    # Image extraction
    min_image_px: int = 100

    # Text filters
    min_page_chars:    int = 40
    max_chars_in_prompt: int = 2000

    # Evaluation
    eval_n_per_geo:  int  = 10
    eval_seed:       int  = 42


settings = Settings()
settings.data_dir.mkdir(parents=True, exist_ok=True)
settings.storage_dir.mkdir(parents=True, exist_ok=True)
settings.images_dir.mkdir(parents=True, exist_ok=True)

print("✓ Settings ready")
print(f"   VLM         : {settings.vlm_model}")
print(f"   Data dir    : {settings.data_dir}")
print(f"   Storage dir : {settings.storage_dir}")

✓ Settings ready
   VLM         : Qwen/Qwen2.5-VL-7B-Instruct
   Data dir    : /content/data
   Storage dir : /content/storage


---
## Step 4 — Helper functions

In [9]:
def normalize_geo(value: str) -> str:
    value = value.strip().upper()
    if value not in settings.geos:
        raise ValueError(f"Unknown region: {value}")
    return value

def detect_geos_from_query(query: str) -> List[str]:
    text = query.upper()
    return [g for g in settings.geos if re.search(rf"\\b{g}\\b", text)]

def make_doc_id(path: Path) -> str:
    return hashlib.md5(str(path.resolve()).encode()).hexdigest()

def clean_text(text: str) -> str:
    text = text.replace("\\x00", " ")
    text = re.sub(r"[ \\t]+", " ", text)
    text = re.sub(r"\\n{3,}", "\\n\\n", text)
    return text.strip()

def page_has_usable_text(text: str) -> bool:
    return len(clean_text(text)) >= settings.min_page_chars

def build_chunk_id(source: str, page: Optional[int], idx: int) -> str:
    return hashlib.md5(f"{source}|{page}|{idx}".encode()).hexdigest()

def make_image_save_path(pdf_path: Path, page_index: int, img_index: int) -> Path:
    uid = hashlib.md5(
        f"{pdf_path.resolve()}|{page_index}|{img_index}".encode()
    ).hexdigest()
    return settings.images_dir / f"{uid}.png"

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("✓ Helper functions ready")

✓ Helper functions ready


---
## Step 5 — Text and table detection

In [10]:
def is_table_line(line: str) -> bool:
    s = line.strip()
    return s.startswith("|") and s.count("|") >= 2

def split_page_into_segments(markdown_text: str) -> List[Dict]:
    segments: List[Dict] = []
    buffer: List[str] = []
    in_table = False

    for line in markdown_text.split("\\n"):
        if is_table_line(line):
            if not in_table:
                content = "\\n".join(buffer).strip()
                if content:
                    segments.append({"type": "text", "content": content})
                buffer = []
                in_table = True
            buffer.append(line)
        else:
            if in_table:
                content = "\\n".join(buffer).strip()
                if content:
                    segments.append({"type": "table", "content": content})
                buffer = []
                in_table = False
            buffer.append(line)

    if buffer:
        content = "\\n".join(buffer).strip()
        if content:
            segments.append({"type": "table" if in_table else "text", "content": content})
    return segments

def extract_json_blocks(text: str) -> Tuple[List[str], str]:
    pattern = re.compile(r"\\{[^{}]*\\}", re.DOTALL)
    found: List[str] = []

    def replacer(m):
        raw = m.group(0)
        try:
            json.loads(raw)
            found.append(raw)
            return " "
        except (json.JSONDecodeError, ValueError):
            return raw

    cleaned = pattern.sub(replacer, text).strip()
    return found, cleaned

def json_to_sentences(json_str: str) -> str:
    try:
        data = json.loads(json_str)
        if isinstance(data, dict):
            parts = [f"{str(k).replace('_', ' ')}: {v}" for k, v in data.items()]
            return ". ".join(parts) + "."
        if isinstance(data, list):
            return " | ".join(
                json_to_sentences(json.dumps(i)) if isinstance(i, dict) else str(i)
                for i in data
            )
        return str(data)
    except (json.JSONDecodeError, ValueError):
        return json_str

print("✓ Text detection ready")

✓ Text detection ready


---
## Step 6 — Image extraction

In [11]:
def extract_and_save_images(pdf_path: Path, page_index: int) -> List[Dict]:
    """Extract images from PDF page and save as PNG."""
    results: List[Dict] = []
    try:
        doc = fitz.open(str(pdf_path))
        page = doc[page_index]

        for img_idx, img_info in enumerate(page.get_images(full=True)):
            xref = img_info[0]
            try:
                base_img = doc.extract_image(xref)
                w = base_img.get("width", 0)
                h = base_img.get("height", 0)

                if w < settings.min_image_px or h < settings.min_image_px:
                    continue

                pil_img = Image.open(io.BytesIO(base_img["image"]))
                if pil_img.mode not in ("RGB", "L"):
                    pil_img = pil_img.convert("RGB")

                img_save_path = make_image_save_path(pdf_path, page_index, img_idx)
                pil_img.save(str(img_save_path))

                caption = (
                    f"Image from page {page_index} of {pdf_path.name}. "
                    f"Size {w}x{h}px. Visual content: chart, diagram, table, or illustration."
                )

                results.append({
                    "caption": caption,
                    "image_path": str(img_save_path),
                })

            except Exception:
                continue

        doc.close()
    except Exception as e:
        print(f"  ⚠  Image extraction error: {e}")

    return results

print("✓ Image extraction ready")

✓ Image extraction ready


---
## Step 7 — PDF loader

In [12]:
def build_documents_from_pdf(pdf_path: Path, geo: str) -> List[Document]:
    doc_id = make_doc_id(pdf_path)
    all_docs: List[Document] = []

    try:
        pages = pymupdf4llm.to_markdown(str(pdf_path), page_chunks=True)
    except Exception as e:
        print(f"  ✗ Cannot parse {pdf_path.name}: {e}")
        return []

    for page_data in pages:
        page_num = page_data.get("metadata", {}).get("page", 0)
        page_text = page_data.get("text", "") or ""

        base_meta = {
            "geo": geo,
            "source": str(pdf_path),
            "doc_id": doc_id,
            "file_name": pdf_path.name,
            "page": page_num,
        }

        # Extract images from every page
        for item in extract_and_save_images(pdf_path, page_num):
            all_docs.append(Document(
                page_content=item["caption"],
                metadata={
                    **base_meta,
                    "element_type": "vlm_image",
                    "image_path": item["image_path"],
                },
            ))

        # Process text
        if not page_has_usable_text(page_text):
            continue

        for seg in split_page_into_segments(page_text):
            content = seg["content"]
            if not content.strip():
                continue

            if seg["type"] == "table":
                all_docs.append(Document(
                    page_content=clean_text(content),
                    metadata={**base_meta, "element_type": "table"},
                ))
            else:
                json_blocks, remaining = extract_json_blocks(content)

                for raw_json in json_blocks:
                    readable = json_to_sentences(raw_json)
                    if len(readable) >= settings.min_page_chars:
                        all_docs.append(Document(
                            page_content=readable,
                            metadata={**base_meta, "element_type": "json"},
                        ))

                remaining = clean_text(remaining)
                if page_has_usable_text(remaining):
                    all_docs.append(Document(
                        page_content=remaining,
                        metadata={**base_meta, "element_type": "text"},
                    ))

    return all_docs

print("✓ PDF loader ready")

✓ PDF loader ready


---
## Step 8 — Load all PDFs

In [13]:
geo_page_docs: Dict[str, List[Document]] = {}

for geo in settings.geos:
    print(f"\n── {geo} ──")
    geo_dir = settings.data_dir / geo

    if not geo_dir.exists():
        print(f"  ⚠  Not found: {geo_dir}")
        geo_page_docs[geo] = []
        continue

    pdf_paths = sorted(geo_dir.rglob("*.pdf"))
    if not pdf_paths:
        print(f"  ⚠  No PDFs found")
        geo_page_docs[geo] = []
        continue

    all_docs = []
    image_count = 0

    for i, pdf_path in enumerate(pdf_paths, 1):
        size_mb = pdf_path.stat().st_size / (1024 * 1024)
        if size_mb > 100:
            print(f"  [{i}/{len(pdf_paths)}] {pdf_path.name} — SKIP (too large)")
            continue

        print(f"  [{i}/{len(pdf_paths)}] {pdf_path.name}...", end=" ", flush=True)

        try:
            docs = build_documents_from_pdf(pdf_path, geo)
            all_docs.extend(docs)

            counts = {}
            for d in docs:
                t = d.metadata.get("element_type", "?")
                counts[t] = counts.get(t, 0) + 1
                if t == "vlm_image":
                    image_count += 1

            print(f"OK  {counts}")

        except Exception as e:
            print(f"SKIP ({type(e).__name__})")
            continue

        clear_memory()

    geo_page_docs[geo] = all_docs
    print(f"  Done: {len(all_docs)} elements, {image_count} images")

saved_images = list(settings.images_dir.glob("*.png"))
print(f"\n✓ PDFs loaded — {len(saved_images)} images saved to disk")


── APAC ──
  [1/5] copy-protected.pdf... === Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'text': 2}
  [2/5] embedded-images-tables.pdf... === Document parser messages ===
                                                            Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'vlm_image': 1, 'text': 1}
  [3/5] embedded-images.pdf... === Document parser messages ===
                                                                                                                        Using Tesseract for OCR processing.
OCR on page.number=0/1.

OK  {'vlm_image': 3, 'text': 1}
  [4/5] embedded-link.pdf... === Document parser messages ===
                                                                                                                                                                                    Using Tesseract for OCR processing.

OK  {'text': 1}
  [5/5] invalid-pdf-structure-pdfminer-one-page.pdf.

---
## Step 9 — Chunk documents

In [14]:
def chunk_documents(docs: List[Document]) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
    )
    chunked: List[Document] = []
    idx = 0

    for doc in docs:
        elem_type = doc.metadata.get("element_type", "text")
        sub_chunks = splitter.split_documents([doc]) if elem_type == "text" else [doc]

        for chunk in sub_chunks:
            chunk.metadata["chunk_id"] = build_chunk_id(
                chunk.metadata.get("source", ""),
                chunk.metadata.get("page"),
                idx,
            )
            chunked.append(chunk)
            idx += 1

    return chunked

geo_chunk_docs: Dict[str, List[Document]] = {}
for geo in settings.geos:
    chunks = chunk_documents(geo_page_docs[geo])
    geo_chunk_docs[geo] = chunks
    breakdown = {}
    for c in chunks:
        t = c.metadata.get("element_type", "?")
        breakdown[t] = breakdown.get(t, 0) + 1
    print(f"{geo}: {len(chunks)} chunks  {breakdown}")

print("\n✓ Chunking complete")

APAC: 51 chunks  {'text': 35, 'vlm_image': 16}
EMEA: 99 chunks  {'text': 95, 'table': 3, 'vlm_image': 1}
AMER: 11 chunks  {'vlm_image': 2, 'text': 9}

✓ Chunking complete


---
## Step 10 — Build indexes

In [15]:
class BGEEmbeddings(HuggingFaceEmbeddings):
    def embed_query(self, text: str) -> List[float]:
        return super().embed_query(settings.bge_query_prefix + text)

print(f"Loading: {settings.embedding_model}")
embeddings = BGEEmbeddings(
    model_name=settings.embedding_model,
    encode_kwargs={"normalize_embeddings": True},
)

print("Loading reranker...")
reranker = CrossEncoder(settings.reranker_model, max_length=512)

def geo_index_path(geo: str) -> Path:
    return settings.storage_dir / f"faiss_{geo.lower()}"

def build_and_save_indexes(geo_chunks, embed_model):
    faiss_stores, bm25_stores = {}, {}

    for geo, chunks in geo_chunks.items():
        if not chunks:
            print(f"⚠  Skipping {geo}: no chunks")
            continue

        save_path = geo_index_path(geo)
        save_path.mkdir(parents=True, exist_ok=True)

        print(f"\n── {geo} ({len(chunks)} chunks) ──")

        print("  Building FAISS...")
        store = FAISS.from_documents(chunks, embed_model)
        store.save_local(str(save_path))
        faiss_stores[geo] = store

        bm25 = BM25Retriever.from_documents(chunks)
        bm25.k = settings.bm25_top_k
        bm25_stores[geo] = bm25

        with open(save_path / "chunks.pkl", "wb") as f:
            pickle.dump(chunks, f)

        print(f"  ✓ FAISS + BM25 ready")
        clear_memory()

    return faiss_stores, bm25_stores

geo_vectorstores, geo_bm25_stores = build_and_save_indexes(geo_chunk_docs, embeddings)
print("\n✓ Indexes built and saved to Google Drive")

Loading: BAAI/bge-large-en-v1.5


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]


── APAC (51 chunks) ──
  Building FAISS...
  ✓ FAISS + BM25 ready

── EMEA (99 chunks) ──
  Building FAISS...
  ✓ FAISS + BM25 ready

── AMER (11 chunks) ──
  Building FAISS...
  ✓ FAISS + BM25 ready

✓ Indexes built and saved to Google Drive


---
## Step 11 — Load VLM (7B for Colab)

In [17]:
# print(f"Loading VLM: {settings.vlm_model}")
# print("Mode: 4-bit quantized (8 GB VRAM)\n")

# clear_memory()

# bnb = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     llm_int8_enable_fp32_cpu_offload=True,
# )

# vlm = Qwen2VLForConditionalGeneration.from_pretrained(
#     settings.vlm_model,
#     quantization_config=bnb,
#     device_map="auto",
#     torch_dtype=torch.float16,
# )
# vlm_processor = AutoProcessor.from_pretrained(settings.vlm_model)

# print(f"✓ VLM loaded")
# print(f"  Device: {next(vlm.parameters()).device}")
# print(f"  Mode  : 4-bit quantized")

import os
from pathlib import Path

# Use Drive cache
cache_dir = Path("/content/gdrive/MyDrive/model_cache")
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(cache_dir)

print(f"Loading VLM from cache: {cache_dir}")
print("(First time: ~5 min download. After: ~30 sec load)\n")

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

vlm = Qwen2VLForConditionalGeneration.from_pretrained(
    settings.vlm_model,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
    cache_dir=str(cache_dir),  # ← explicit cache
)
vlm_processor = AutoProcessor.from_pretrained(
    settings.vlm_model,
    cache_dir=str(cache_dir),  # ← explicit cache
)

print(f"✓ VLM loaded (from cache)")

Loading VLM from cache: /content/gdrive/MyDrive/model_cache
(First time: ~5 min download. After: ~30 sec load)



config.json: 0.00B [00:00, ?B/s]

You are using a model of type qwen2_5_vl to instantiate a model of type qwen2_vl. This is not supported for all configurations of models and can yield errors.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✓ VLM loaded (from cache)


---
## Step 12 — VLM image description (cached)

In [18]:
_image_caption_cache: Dict[str, str] = {}

def describe_image_with_vlm(image_path: str) -> str:
    """Describe image with VLM. Results cached per session."""
    if image_path in _image_caption_cache:
        return _image_caption_cache[image_path]

    img_path_obj = Path(image_path)
    if not img_path_obj.exists():
        return "[Image file not found]"

    try:
        pil_img = Image.open(image_path)
        if pil_img.mode not in ("RGB", "L"):
            pil_img = pil_img.convert("RGB")

        prompt = (
            "Analyze this image in detail.\n"
            "- Type of content (table, chart, diagram, etc.)\n"
            "- Main findings and key numbers\n"
            "- All visible labels and text\n"
            "- Trends and patterns\n"
            "Be specific and comprehensive."
        )

        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": pil_img},
                {"type": "text", "text": prompt},
            ],
        }]

        text_input = vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = vlm_processor(
            text=[text_input],
            images=[pil_img],
            return_tensors="pt",
            padding=True,
        ).to(vlm.device)

        with torch.no_grad():
            output_ids = vlm.generate(
                **inputs,
                max_new_tokens=settings.image_max_tokens,
                do_sample=False,
                temperature=settings.temperature,
            )

        generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
        description = vlm_processor.batch_decode(
            generated_ids, skip_special_tokens=True
        )[0].strip()

        _image_caption_cache[image_path] = description
        return description

    except Exception as e:
        error_msg = f"[Error: {str(e)[:50]}]"
        _image_caption_cache[image_path] = error_msg
        return error_msg

print("✓ Image description function ready (with caching)")

✓ Image description function ready (with caching)


---
## Step 13 — Retrieval and generation

In [19]:
SYSTEM_PROMPT = (
    "You are a sales assistant. Answer using ONLY the provided context. "
    "Do not use external knowledge. "
    "If the answer is not in the context, say: "
    "'I do not have the answer in the provided documents.'"
)

def format_text_context(docs: List[Document]) -> str:
    blocks = []
    for i, doc in enumerate(docs, 1):
        meta = doc.metadata
        content = doc.page_content[:settings.max_chars_in_prompt]
        blocks.append(
            f"[{i}] {meta.get('element_type','?')} | "
            f"{meta.get('file_name','?')} p.{meta.get('page','?')}\n{content}"
        )
    return "\n\n---\n\n".join(blocks)

def run_vlm_text_only(messages: list) -> str:
    text_input = vlm_processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = vlm_processor(
        text=[text_input],
        return_tensors="pt",
        padding=True,
    ).to(vlm.device)

    with torch.no_grad():
        output_ids = vlm.generate(
            **inputs,
            max_new_tokens=settings.max_new_tokens,
            do_sample=settings.do_sample,
        )

    generated_ids = output_ids[:, inputs.input_ids.shape[1]:]
    return vlm_processor.batch_decode(
        generated_ids, skip_special_tokens=True
    )[0].strip()

def get_candidates(question: str, faiss_stores, bm25_stores):
    regions = detect_geos_from_query(question)
    targets = regions if regions else list(faiss_stores.keys())

    seen, candidates = set(), []

    for geo in targets:
        if geo in faiss_stores:
            for doc, score in faiss_stores[geo].similarity_search_with_score(
                question, k=settings.retrieval_fetch_k
            ):
                if settings.score_threshold is None or score <= settings.score_threshold:
                    cid = doc.metadata.get("chunk_id", "")
                    if cid not in seen:
                        seen.add(cid)
                        candidates.append(doc)
        if geo in bm25_stores:
            for doc in bm25_stores[geo].invoke(question):
                cid = doc.metadata.get("chunk_id", "")
                if cid not in seen:
                    seen.add(cid)
                    candidates.append(doc)

    return targets, candidates

def rerank_candidates(question: str, candidates: List[Document]):
    if not candidates:
        return []

    pairs = [[question, doc.page_content[:512]] for doc in candidates]
    scores = reranker.predict(pairs).tolist()
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return ranked[:settings.top_k]

def answer_question(question: str, faiss_stores, bm25_stores) -> Dict:
    regions, candidates = get_candidates(question, faiss_stores, bm25_stores)
    top_scored = rerank_candidates(question, candidates)

    if not top_scored:
        return {
            "question": question,
            "answer": "I do not have the answer in the provided documents.",
            "geo_used": regions,
            "mode": "no_match",
            "sources": [],
        }

    top_docs = [doc for doc, _ in top_scored]
    geo_label = ", ".join(regions) if regions else "ALL"

    # Collect image descriptions
    image_descriptions: List[str] = []
    for doc in top_docs:
        img_path = doc.metadata.get("image_path")
        if not img_path or not Path(img_path).exists():
            continue

        description = describe_image_with_vlm(img_path)
        image_descriptions.append(
            f"[Image from {doc.metadata.get('file_name','?')} "
            f"p.{doc.metadata.get('page','?')}]\n{description}"
        )

    # Build context
    text_context = format_text_context(top_docs)
    if image_descriptions:
        img_block = "\n\n".join(image_descriptions)
        full_context = f"{text_context}\n\n=== Images ===\n{img_block}"
        mode = "vision"
    else:
        full_context = text_context
        mode = "text"

    # Generate answer (text-only mode)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Region: {geo_label}\n\n"
                f"Context:\n{full_context}\n\n"
                f"Question: {question}"
            ),
        },
    ]

    answer = run_vlm_text_only(messages)

    sources = [
        {
            "type": d.metadata.get("element_type"),
            "file": d.metadata.get("file_name"),
            "page": d.metadata.get("page"),
            "score": round(float(s), 3),
            "has_image": bool(d.metadata.get("image_path")),
        }
        for d, s in top_scored
    ]

    return {
        "question": question,
        "answer": answer,
        "geo_used": regions,
        "mode": mode,
        "sources": sources,
    }

print("✓ Pipeline ready")

✓ Pipeline ready


---
## Step 14 — Test queries

In [20]:
def print_result(result: Dict) -> None:
    print(f"Question: {result['question']}")
    print(f"Mode    : {result['mode'].upper()}")
    print(f"\nAnswer:\n{result['answer']}")
    print()
    if result['sources']:
        print("Sources:")
        for s in result['sources']:
            img = " [img]" if s['has_image'] else ""
            print(f"  {s['type']:8} {s['file']} p.{s['page']} score={s['score']:+.2f}{img}")
    print()

# Query 1
result = answer_question(
    "What is the refund policy for APAC?",
    geo_vectorstores, geo_bm25_stores,
)
print_result(result)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Question: What is the refund policy for APAC?
Mode    : TEXT

Answer:
I do not have the answer in the provided documents.

Sources:
  text     invalid-pdf-structure-pdfminer-one-page.pdf p.0 score=+0.00
  text     invalid-pdf-structure-pdfminer-one-page.pdf p.0 score=+0.00
  text     multi-column.pdf p.0 score=+0.00



In [21]:
# Query 2 — with images
result = answer_question(
    "Describe charts and diagrams in the documents of APAC",
    geo_vectorstores, geo_bm25_stores,
)
print_result(result)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/nn/modules.py:409: UserWarning: FP4 quantization state not initialized. Please call .cuda() or .to(device) on the LinearFP4 layer first.
  warnings.warn(


Question: Describe charts and diagrams in the documents of APAC
Mode    : VISION

Answer:
The context does not provide information about charts and diagrams specifically for the APAC region. The documents listed refer to pages from PDFs with titles like "a1977-backus-p21.pdf," "table-multi-row-column-cells.pdf," and "invalid-pdf-structure-pdfminer-one-page.pdf." However, there is no indication that these documents are related to the APAC region. Therefore, I do not have the answer in the provided documents regarding charts and diagrams in the documents of the APAC region.

Sources:
  vlm_image a1977-backus-p21.pdf p.0 score=+0.01 [img]
  vlm_image table-multi-row-column-cells.pdf p.0 score=+0.01 [img]
  vlm_image invalid-pdf-structure-pdfminer-one-page.pdf p.0 score=+0.01 [img]



In [23]:
# Query 3
result = answer_question(
   "Summary of all the the documents of APAC documents",
    geo_vectorstores, geo_bm25_stores,
)
print_result(result)

Question: Summary of all the the documents of APAC documents
Mode    : VISION

Answer:
The provided documents do not contain any information related to the Asia-Pacific (APAC) region. The documents focus on the Eastern Mediterranean region, specifically mentioning Israel, Egypt, and the natural gas production there. There is no mention of APAC in these documents. Therefore, I do not have the answer in the provided documents.

Sources:
  text     chevron-page.pdf p.0 score=+0.00
  vlm_image a1977-backus-p21.pdf p.0 score=+0.00 [img]
  text     invalid-pdf-structure-pdfminer-one-page.pdf p.0 score=+0.00



In [24]:
# Query 3
result = answer_question(
   "take any image and explain what it represent",
    geo_vectorstores, geo_bm25_stores,
)
print_result(result)

Question: take any image and explain what it represnt
Mode    : TEXT

Answer:
The document you've provided discusses "Layout Parser," which is described as a unified toolkit for deep learning-based document image analysis. It mentions recent advancements in document image analysis (DIA) that have been driven by neural networks. The authors highlight the challenges of deploying research outcomes in production due to factors such as loosely organized codebases and sophisticated model configurations. They also point out that while there have been efforts to improve reusability and simplify deep learning model development in fields like natural language processing and computer vision, these efforts are not optimized for the specific challenges in the domain of DIA. The document emphasizes this as a significant gap in existing toolkits, as DIA is central to academic research across various disciplines in the social sciences and humanities.

Sources:
  text     layout-parser-paper-fast.pdf p

---
## Key differences: Colab vs Mac

| Feature | Mac | Colab |
|---|---|---|
| GPU VRAM | 8-12 GB | 16+ GB |
| VLM size | 3B | 7B (better) |
| Quantization | 8-bit | 4-bit |
| Image mode | Text-only | Text+images |
| Storage | Local | Google Drive |
| Speed | Slow | Fast |

All features work on both. Colab is 10-100x faster.

---

## To use your own PDFs

1. Create folder in Google Drive:
   ```
   Google Drive/RAG_data/data/APAC/
   Google Drive/RAG_data/data/EMEA/
   Google Drive/RAG_data/data/AMER/
   ```

2. Upload your PDFs to these folders

3. Run Steps 0-14

All storage is persistent in Google Drive.
